In [8]:
"""
Задача 2: Програматор EEPROM польотного контролера БПЛА.
Вихідний файл: eeprom_pid.bin (заголовок '<4sHH' = 8 байтів, вісь '<fff' = 12 байтів).
"""
import struct

HEADER_FORMAT = '<4sHH'
AXIS_FORMAT = '<3f'
HEADER_SIZE = struct.calcsize(HEADER_FORMAT)  # 8 байтів
AXIS_SIZE = struct.calcsize(AXIS_FORMAT)      # 12 байтів

def init_eeprom(filepath, axes_gains):
    with open(filepath, 'wb', buffering=0) as file:
        file.write(struct.pack(HEADER_FORMAT, b'FCv2', 1, 3))

        for axis in axes_gains:
            file.write(struct.pack(AXIS_FORMAT, *axis))

def patch_axis_gains(filepath, axis_index, limiter_callback):
    with open(filepath, 'r+b', buffering=0) as file:
        file.seek(8 + 12 * axis_index)
        axis_gains = list(struct.unpack(AXIS_FORMAT, file.read(AXIS_SIZE)))

        for i in range(0, len(axis_gains), 1):
            axis_gains[i] = limiter_callback(axis_gains[i])

        file.seek(8 + 12 * axis_index)
        file.write(struct.pack(AXIS_FORMAT, *axis_gains))

def main():
    filepath = '../output/eeprom_pid.bin'
    initial_gains = [
        (4.2, 0.85, 0.12),   # 0: Roll
        (10.8, 1.40, 0.25),  # 1: Pitch
        (6.5, 0.50, 0.05),   # 2: Yaw
    ]

    # 1. Ініціалізація EEPROM
    init_eeprom(filepath, initial_gains)

    # 2. Оновлення осі Pitch (axis_index=1) через callback-лямбду
    patch_axis_gains(filepath, 1, lambda g: min(round(g * 1.15, 4), 12.0))

    # 3. Зчитування всіх осей у зворотному порядку
    with open(filepath, 'rb', buffering=0) as file:
        sign, ver, q_axis = struct.unpack(HEADER_FORMAT, file.read(HEADER_SIZE))
        print(sign, ver, q_axis)
        for i in range(2, -1, -1):
            file.seek(8 + 12 * i)
            axis = tuple(struct.unpack(AXIS_FORMAT, file.read(AXIS_SIZE)))
            print(axis)

if __name__ == '__main__':
    main()


b'FCv2' 1 3
(6.5, 0.5, 0.05000000074505806)
(12.0, 1.6100000143051147, 0.2874999940395355)
(4.199999809265137, 0.8500000238418579, 0.11999999731779099)
